# Lesson 3: Binding and dose response

**You will:** vary the amount of a ligand, measure how much of a receptor it occupies, and read the dissociation constant `Kd` off the curve.

**The model** ([`binding.bngl`](binding.bngl)): `L + R <-> LR`, binding at `kon` and unbinding at `koff`. The dissociation constant is `Kd = koff / kon` = 100 molecules here.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

import bngsim

## One dose

With 100 ligand molecules and 10 receptors, run until binding settles:

In [ ]:
model = bngsim.Model.load("binding.bngl")
sim = bngsim.Simulator(model, method="ode")
r = sim.run(t_span=(0, 100), n_points=101)
plt.plot(r.time, r.observables["Bound"])
plt.xlabel("time (s)")
plt.ylabel("receptors bound")
plt.show()

## A dose-response curve

Now scan the ligand over five orders of magnitude. For each dose, set `L0`, reset the model to its new starting amounts, and run to equilibrium. The fraction of receptors bound is the *occupancy*.

In [ ]:
kon, koff, R0 = 0.001, 0.1, 10
Kd = koff / kon
doses = np.logspace(0, 5, 21)
occupancy = []
for L in doses:
    model.set_param("L0", L)
    model.reset()
    r = sim.run(t_span=(0, 200), n_points=2)
    occupancy.append(r.observables["Bound"][-1] / R0)
occupancy = np.array(occupancy)

plt.semilogx(doses, occupancy, "o", label="simulated")
plt.semilogx(doses, doses / (doses + Kd), label="L / (L + Kd)")
plt.axvline(Kd, color="gray", linestyle=":", label="Kd")
plt.xlabel("ligand molecules")
plt.ylabel("fraction of receptors bound")
plt.legend()
plt.show()

## Reading the curve

Half the receptors are bound when the free ligand equals `Kd`. The simple formula `L / (L + Kd)` assumes binding uses up no ligand, which is nearly true when ligand far outnumbers receptor. The exact occupancy solves the same quadratic as lesson 1:

In [ ]:
def exact_occupancy(L):
    b = kon * (L + R0) + koff
    return (b - math.sqrt(b * b - 4 * kon * kon * L * R0)) / (2 * kon) / R0

for L, f in zip(doses[::5], occupancy[::5]):
    print(f"L0 = {L:8.0f}: occupancy exact {exact_occupancy(L):.4f}, simulated {f:.4f}, simple formula {L / (L + Kd):.4f}")

## Try it

1. Make the receptor ten times stickier (`koff` ten times smaller). Where does the curve move, left or right?
2. Raise `R0` to 1000, as many receptors as ligand. How far is the simple formula off now, and why?